##Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

###Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max, count, avg, when, round as spark_round
from pyspark.sql import Window
import pyspark.sql.functions as F

### Order_Items Table Data Manipulation and Cleaning

In [0]:
df_order_items_bronze = spark.table("olist_ecommerce_project.bronze.brz_order_items")

# Basic profiling
print("Total rows:", df_order_items_bronze.count())
print("Distinct order_id:", df_order_items_bronze.select("order_id").distinct().count())
print("Distinct product_id:", df_order_items_bronze.select("product_id").distinct().count())
print("Distinct seller_id:", df_order_items_bronze.select("seller_id").distinct().count())

# Null check
df_order_items_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_order_items_bronze.columns
]).show()

# Check for negative or zero prices
print("\nRows with zero or negative price:")
df_order_items_bronze.filter(col("price") <= 0).count()

print("Rows with zero or negative freight:")
df_order_items_bronze.filter(col("freight_value") <= 0).count()

##### Zero/negative freight: 383 rows — needs investigation

In [0]:
# Inspect the 383 zero/negative freight rows
df_order_items_bronze.filter(col("freight_value") <= 0).select(
    "order_id",
    "product_id",
    "price",
    "freight_value"
).show(10, truncate=False)

# Are they all zero or are some actually negative?
print("Zero freight rows:", df_order_items_bronze.filter(col("freight_value") == 0).count())
print("Negative freight rows:", df_order_items_bronze.filter(col("freight_value") < 0).count())

# Check if these zero freight orders are legitimate
# e.g. free shipping promotions — high price items sometimes have free freight
df_order_items_bronze.filter(col("freight_value") == 0).select(
    "price", "freight_value"
).summary("min", "max", "mean").show()

In [0]:
df_orders_silver = spark.table("olist_ecommerce_project.silver.slv_orders")
df_products_silver = spark.table("olist_ecommerce_project.silver.slv_products")

# Check 1: order_ids in Order Items that don't exist in Silver Orders
orphan_orders = df_order_items_bronze.join(
    df_orders_silver.select("order_id"),
    on="order_id",
    how="left_anti"
)
print("Order items with no matching order in slv_orders:", orphan_orders.count())

# Check 2: product_ids in Order Items that don't exist in Silver Products
orphan_products = df_order_items_bronze.join(
    df_products_silver.select("product_id"),
    on="product_id",
    how="left_anti"
)
print("Order items with no matching product in slv_products:", orphan_products.count())

In [0]:
# Step 1: Drop source file audit column
df_order_items_silver = df_order_items_bronze.drop("_source_file")

# Step 2: Add total_item_value column 
# price + freight_value = true cost per item to the customer
df_order_items_silver = df_order_items_silver.withColumn(
    "total_item_value",
    col("price") + col("freight_value")
)

# Sanity check before writing
print("Total rows:", df_order_items_silver.count())
df_order_items_silver.select(
    "order_id",
    "order_item_id",
    "product_id",
    "price",
    "freight_value",
    "total_item_value"
).show(10, truncate=False)

Creating Order_Items as all the data is clear and we have added total_item_value column to check total value

In [0]:
# Round total_item_value to 2 decimal places
df_order_items_silver = df_order_items_silver.withColumn(
    "total_item_value",
    spark_round(col("total_item_value"), 2)
)

# Write to Silver
(
    df_order_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_order_items")
)

print("slv_order_items written successfully")

## Order Items — Silver Layer Cleaning Notes

The Order Items table connects orders to products and sellers, and is the 
primary source for revenue calculations in the Gold layer. Each order can 
have multiple items, so `order_id` is not unique in this table.

### Profiling Summary

| Metric | Value |
|---|---|
| Total rows | 112,650 |
| Distinct order_ids | 98,666 |
| Distinct product_ids | 32,951 |
| Distinct seller_ids | 3,095 |
| Null values | None |

### Checks Performed

**1. Price Validation**
- Checked for zero or negative prices — none found
- All price values are valid and positive

**2. Freight Value Validation**
- Found 383 rows with zero freight value
- Investigated and confirmed these are legitimate free shipping 
  promotions — mid-to-high value items (mean price ~98.6 BRL, 
  max 712.9 BRL) where sellers or Olist offered free freight
- Decision: kept as-is — zero freight is a valid business value, 
  not a data error

**3. Referential Integrity Check**
- Verified every `order_id` in Order Items exists in `slv_orders` → 0 orphans found
- Verified every `product_id` in Order Items exists in `slv_products` → 0 orphans found
- Perfect referential integrity confirmed across all related Silver tables

### Transformations Applied

**1. Dropped `_source_file`**
- Removed Bronze-specific audit column as per Silver layer standard

**2. Added `total_item_value`**
- Computed as `price + freight_value` — represents the true cost 
  per item to the customer including shipping
- Pre-computed here in Silver to avoid repeating this arithmetic 
  in every Gold layer query
- Rounded to 2 decimal places to avoid floating point precision 
  issues (e.g. 218.04000000000002 → 218.04)

### Gold Layer Use Cases

| Column | Gold Use Case |
|---|---|
| `price` | Revenue analysis per product/category/seller |
| `freight_value` | Freight cost analysis and free shipping impact |
| `total_item_value` | Total revenue per order, per seller, per state |
| `order_item_id` | Count items per order for basket size analysis |